# UCS420 Assignment 6
### NumPy II

Gurkirpa Singh, 3Q34, 1024170378

In [1]:
import numpy as np

## Ques 1

The sensor reads 2°C too high on avg, so part (a) adds the correction to the whole
array in one go. I've used the corrected readings for (b) to (d). For (c) the
comparison `temp > 32` gives a True/False mask, and indexing with it keeps only the
True positions. Summing the mask counts them since True is treated as 1.

In [2]:
# Q1
temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])
print("original:", temperature)

temp = temperature + 2
print("a) +2 correction :", temp)

fahr = temp * 9 / 5 + 32
print("b) in fahrenheit :", fahr)

mask = temp > 32
print("c) mask          :", mask)
print("   above 32      :", temp[mask])

print("d) count above 32:", mask.sum())

original: [25 28 31 35 38 27 33 40]
a) +2 correction : [27 30 33 37 40 29 35 42]
b) in fahrenheit : [ 80.6  86.   91.4  98.6 104.   84.2  95.  107.6]
c) mask          : [False False  True  True  True False  True  True]
   above 32      : [33 37 40 35 42]
d) count above 32: 5


e) With a for loop, python handles each value one by one and checks its type
every time, which is slow. A vectorised op like `temp + 2` hands the whole array to
numpy, which runs the loop internally in compiled C over one contiguous block of
memory. So it's much faster, esp. on large data (lakhs of sensor readings).

Boolean indexing does the same for filtering. Instead of an `if` inside a loop and
appending to a list, one comparison builds the mask and one indexing step pulls out
the matches. The code is shorter too and reads close to the actual condition, so
there are fewer places to make mistakes.

## Ques 2

Rows are users and columns are days. `axis=0` adds down the columns, so it gives
one total per day. `axis=1` adds across the rows, giving one total per user.
`argmax` returns a flat index (as if the matrix were one long row), so I use
`np.unravel_index` to turn it back into (row, col).

In [3]:
# Q2
steps = np.array([[5000, 6200, 7100],
                  [8000, 7500, 9000],
                  [4500, 5100, 4800],
                  [9000, 8500, 9500]])
print(steps)

print("a) total :", steps.sum())
print("b) mean  :", steps.mean())
print("c) max   :", steps.max())
print("   min   :", steps.min())

[[5000 6200 7100]
 [8000 7500 9000]
 [4500 5100 4800]
 [9000 8500 9500]]
a) total : 84200
b) mean  : 7016.666666666667
c) max   : 9500
   min   : 4500


In [4]:
print("d) total per day  :", steps.sum(axis=0))
print("e) total per user :", steps.sum(axis=1))

d) total per day  : [26500 27300 30400]
e) total per user : [18300 24500 14400 27000]


In [5]:
idx = steps.argmax()
row, col = np.unravel_index(idx, steps.shape)
print("f) flat index:", idx)
print("   position  :", (int(row), int(col)))
print("   i.e. user", row + 1, "on day", col + 1, "with", steps[row, col], "steps")

f) flat index: 11
   position  : (3, 2)
   i.e. user 4 on day 3 with 9500 steps


## Ques 3

A plain slice in numpy is a view, meaning it points to the same memory as the
original. So changing `subset` changes `original` as well. `.copy()` makes a
separate array, so editing it leaves the original alone.

In [6]:
# Q3 a, b, c
original = np.array([1, 2, 3, 4, 5, 6])
subset = original[1:5]
print("subset  :", subset)

subset[0] = 999
print("after setting subset[0] = 999")
print("original:", original)
print("subset  :", subset)

subset  : [2 3 4 5]
after setting subset[0] = 999
original: [  1 999   3   4   5   6]
subset  : [999   3   4   5]


In [7]:
# Q3 d
original = np.array([1, 2, 3, 4, 5, 6])  # reset so the 999 doesnt carry over
copied = original[1:5].copy()
copied[0] = 500
print("after setting copied[0] = 500")
print("original:", original)
print("copied  :", copied)

after setting copied[0] = 500
original: [1 2 3 4 5 6]
copied  : [500   3   4   5]


For (f), indices start at 0, so "rows 1-2, columns 2-3" is `mat[1:3, 2:4]`,
the bottom right 2x2 block.

In [8]:
# Q3 e, f
mat = np.arange(1, 13).reshape(3, 4)
print(mat)

print("first row     :", mat[0])
print("last row      :", mat[-1])
print("second column :", mat[:, 1])
print("rows 1-2, cols 2-3:")
print(mat[1:3, 2:4])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
first row     : [1 2 3 4]
last row      : [ 9 10 11 12]
second column : [ 2  6 10]
rows 1-2, cols 2-3:
[[ 7  8]
 [11 12]]


`flatten()` always returns a new copy. `ravel()` returns a view whenever it can
(it can here since `mat` is contiguous), so changes to it show up in `mat`.

In [9]:
# Q3 g
flat = mat.flatten()
rav = mat.ravel()
print("flatten:", flat)
print("ravel  :", rav)

flatten: [ 1  2  3  4  5  6  7  8  9 10 11 12]
ravel  : [ 1  2  3  4  5  6  7  8  9 10 11 12]


In [10]:
# Q3 h
rav[0] = 100
print("after rav[0] = 100")
print(mat)

after rav[0] = 100
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


In [11]:
# Q3 i
flat[1] = 200
print("after flat[1] = 200")
print("flat:", flat)
print(mat)

after flat[1] = 200
flat: [  1 200   3   4   5   6   7   8   9  10  11  12]
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


So `mat[0, 0]` became 100 through `ravel`, but the 200 set through `flatten` never reached `mat`.

In [12]:
# Q3 j
print("shape :", mat.shape)
print("ndim  :", mat.ndim)
print("size  :", mat.size)
print("dtype :", mat.dtype)

shape : (3, 4)
ndim  : 2
size  : 12
dtype : int64


## Ques 4

Each row of X is one user and the columns are sleep, activity and stress. The OLS
formula $\beta = (X^TX)^{-1}X^Ty$ gives the weights that make $X\beta$ as close to
$y$ as possible in the least squares sense.

In [13]:
# Q4 a
X = np.array([[6, 70, 3],
              [5, 50, 6],
              [8, 80, 2],
              [4, 30, 8]])
y = np.array([40, 65, 30, 85])

print("shape:", X.shape)
print("ndim :", X.ndim)

shape: (4, 3)
ndim : 2


In [14]:
# Q4 b
print(X.T)
print("shape:", X.T.shape)

[[ 6  5  8  4]
 [70 50 80 30]
 [ 3  6  2  8]]
shape: (3, 4)


The transpose swaps rows and columns. X is 4x3 (users x features), X.T is 3x4,
so each row is now one feature across all users. Row 0 is everyone's sleep hours,
row 1 their activity and row 2 their stress.

In [15]:
# Q4 c
xtx = X.T @ X
print(xtx)

[[  141  1430    96]
 [ 1430 14700   910]
 [   96   910   113]]


X itself is 4x3 and not square, so it can't be inverted. `X.T @ X` is 3x3, and it
is invertible as long as its determinant isn't 0, so I check that first.

In [16]:
# Q4 d
det = np.linalg.det(xtx)
print("det:", det)

if np.isclose(det, 0):
    print("singular, no inverse")
else:
    xtx_inv = np.linalg.inv(xtx)
    print(xtx_inv)

det: 753700.000000011
[[ 1.10521428 -0.09848746 -0.14581398]
 [-0.09848746  0.00891203  0.01190129]
 [-0.14581398  0.01190129  0.0368847 ]]


In [17]:
# Q4 e
beta = xtx_inv @ X.T @ y
print("beta     :", beta)
print("predicted:", X @ beta)
print("actual   :", y)

# cross check with numpy's own solver
print("lstsq    :", np.linalg.lstsq(X, y, rcond=None)[0])

beta     : [-0.15921454  0.14083853 10.06302242]
predicted: [39.09247711 66.62398832 30.11941091 84.09247711]
actual   : [40 65 30 85]
lstsq    : [-0.15921454  0.14083853 10.06302242]


f) The model is score approx. = b1*sleep + b2*activity + b3*stress. Each coefficient
is how much the predicted score moves when that feature goes up by 1 and the other
two stay fixed.

b1 ≈ -0.16 for sleep, so more sleep slightly lowers the score, i.e. a well rested
user needs a bit less help. b2 ≈ 0.14 for activity looks tiny, but activity ranges
from 30 to 80 so it still adds about 4 to 11 points. b3 ≈ 10.06 for stress is the
main driver, every extra stress point adds roughly 10. That fits the data, the user
with stress 8 has the highest score (85) and the one with stress 2 the lowest (30).

There's no intercept term here since X has no column of ones, and with just 4 users
for 3 weights the fit is very tight, so I wouldn't read too much into it.

In [18]:
# Q4 g
new_user = np.array([5, 40, 7])
pred = new_user @ beta
print("predicted score for new user:", round(pred, 2))

predicted score for new user: 75.28
